# Study 842 — Implementation Shortfall — the teardown

The dollar-neutral cross-sectional book, the linear + super-linear (participation) impact cost model, the cost ladder, the deceptive linear break-even, the turnover curve, HAC inference, and the 20-seed synthetic control. Synthetic-only, capped at `NONE`.

In [1]:
R = {'fp': '79082a088002', 'null_fp': '87551053f9a3', 'as_of': '2026-06-30', 'n_names': 30, 'n_days': 2519, 'edge': 0.0005, 'phi': 0.96, 'frac': 0.2, 'seed': 842, 'turnover': 0.348, 'gross_bps': 12.25, 'gross_t': 7.65, 'gross_sharpe': 2.27, 'gross_ann': 30.9, 'ladder': [('paper (0 cost)', 0, 0, 2.27, 2.27, 12.25, 0.0, 7.65, 30.9), ('optimistic', 5, 20, 2.27, 1.39, 7.51, 4.74, 4.69, 18.9), ('realistic', 10, 50, 2.27, 0.24, 1.27, 10.98, 0.79, 3.2), ('stressed', 20, 100, 2.27, -1.78, -9.71, 21.96, -5.9, -24.5)], 'breakeven_linear': 35.24, 'curve': [(0.995, 0.152, 2.06, 1.4, 72.0), (0.98, 0.254, 2.15, 0.86, 45.9), (0.96, 0.348, 2.27, 0.24, 35.2), (0.9, 0.533, 2.17, -1.73, 22.4), (0.7, 0.893, 2.21, -6.77, 13.8), (0.3, 1.343, 2.46, -15.54, 10.0)], 'null_mean_t': 0.01, 'null_sd_t': 0.92, 'null_fire': 1, 'plant_mean_t': 6.07, 'plant_sd_t': 0.92, 'plant_fire': 20}

## The paper portfolio — the gross (0-cost) edge

Dollar-neutral long-top-20% / short-bottom-20% on the signal known at close `t-1`; 30 names, 2519 days, moderate turnover (0.348/day).

In [2]:
print(f"gross spread : {R['gross_bps']:+.2f} bps/day  NW(10) t = {R['gross_t']:+.2f}")
print(f"gross Sharpe : {R['gross_sharpe']:.2f}  (~{R['gross_ann']:+.1f}%/yr)")
print(f"turnover     : {R['turnover']:.3f}/day (one-way)")

gross spread : +12.25 bps/day  NW(10) t = +7.65
gross Sharpe : 2.27  (~+30.9%/yr)
turnover     : 0.348/day (one-way)


## The cost ladder — same book, four cost worlds

`net = gross - turnover·cost_bps - impact_coef_bps·turnover²` — a linear one-way cost plus a super-linear market-impact term (~ participation).

In [3]:
print(f"{'scenario':<16}{'one-way/impact':>16}{'gross Sh':>10}{'net Sh':>9}{'net bps':>9}{'cost/day':>10}{'net t':>8}")
for lab, cb, ic, gs, ns, nb_, cd, nt, ann in R['ladder']:
    print(f"{lab:<16}{f'{cb:g}bp / {ic:g}':>16}{gs:>10.2f}{ns:>9.2f}{nb_:>+9.2f}{cd:>10.2f}{nt:>+8.2f}")

scenario          one-way/impact  gross Sh   net Sh  net bps  cost/day   net t
paper (0 cost)           0bp / 0      2.27     2.27   +12.25      0.00   +7.65
optimistic              5bp / 20      2.27     1.39    +7.51      4.74   +4.69
realistic              10bp / 50      2.27     0.24    +1.27     10.98   +0.79
stressed              20bp / 100      2.27    -1.78    -9.71     21.96   -5.90


## The break-even cost — the deceptive 'headroom'

The break-even *linear* one-way cost (net alpha → 0) looks comfortable — but it **ignores market impact**. Quoting a break-even without an impact model is another way of ignoring costs.

In [4]:
print(f"break-even LINEAR one-way cost : {R['breakeven_linear']:.2f} bps")
print(f"realistic all-in cost charged  : ~{R['ladder'][2][6]:.1f} bps/day")
print("the 35 bp 'headroom' is a fantasy: with impact, the realistic 11 bps/day "
      "cost already matches the 12.25 bps/day gross edge -> net ~0")

break-even LINEAR one-way cost : 35.24 bps
realistic all-in cost charged  : ~11.0 bps/day
the 35 bp 'headroom' is a fantasy: with impact, the realistic 11 bps/day cost already matches the 12.25 bps/day gross edge -> net ~0


## The turnover curve — alpha dies as a FUNCTION of turnover

Hold the gross edge fixed (same `edge`), turn only the persistence knob φ; charge the **realistic** cost at every point. Gross Sharpe is ~flat; net Sharpe falls off a cliff.

In [5]:
print(f"{'phi':>6}{'turnover':>10}{'gross Sh':>10}{'net Sh':>9}{'break-even':>12}")
for phi, tu, gs, ns, be in R['curve']:
    print(f"{phi:>6}{tu:>10.3f}{gs:>10.2f}{ns:>9.2f}{be:>10.1f}bp")
print('gross Sharpe range:', round(max(c[2] for c in R['curve'])-min(c[2] for c in R['curve']),2),
      '| net Sharpe range:', round(max(c[3] for c in R['curve'])-min(c[3] for c in R['curve']),2))

   phi  turnover  gross Sh   net Sh  break-even
 0.995     0.152      2.06     1.40      72.0bp
  0.98     0.254      2.15     0.86      45.9bp
  0.96     0.348      2.27     0.24      35.2bp
   0.9     0.533      2.17    -1.73      22.4bp
   0.7     0.893      2.21    -6.77      13.8bp
   0.3     1.343      2.46   -15.54      10.0bp
gross Sharpe range: 0.4 | net Sharpe range: 16.94


## Synthetic control — the machinery is unbiased (20 seeds)

Live: the gross book must recover the planted edge and stay silent on the null. A faithful-engine check only — never cited in support of a stamp.

In [6]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from cost_gap import data, strategy as st
null = st.seed_robust_control(data, edge=0.0, n_seeds=8, n_days=1200)
plant = st.seed_robust_control(data, edge=0.0005, n_seeds=8, n_days=1200)
print(f"null   (edge=0)     : gross t mean {null['mean_t']:+.2f} (sd {null['sd_t']:.2f}), "
      f"|t|>=2 in {null['fire_count']}/{null['n_seeds']}")
print(f"planted (edge=5e-4) : gross t mean {plant['mean_t']:+.2f} (sd {plant['sd_t']:.2f}), "
      f"|t|>=2 in {plant['fire_count']}/{plant['n_seeds']}")
print('(frozen 20-seed run: null 1/20 at t~0.0; planted 20/20 at t~6.1)')

null   (edge=0)     : gross t mean +0.03 (sd 0.79), |t|>=2 in 0/8
planted (edge=5e-4) : gross t mean +5.48 (sd 0.67), |t|>=2 in 8/8
(frozen 20-seed run: null 1/20 at t~0.0; planted 20/20 at t~6.1)


## Verdict

- **Signal — NONE.** A synthetic-only method demo: the gross edge is *planted*, not found on a real tape, so it can never earn `REAL` (which needs a robust *t* ≥ 2 on real data). The 0-cost gross Sharpe is a genuine **2.27** (NW *t* = **+7.65**) — but that is the paper number.
- **Tradability — MIRAGE.** Net Sharpe falls **2.27 → 1.39 → 0.24** (net *t* = 0.79, dead) → **-1.78** down the cost ladder, and reaches **-15.54** at high turnover. Nothing survives the trading.
- **Does ignoring costs manufacture the edge? — CONFIRMED.** The entire 30.9%/yr paper triumph is the cost of trading the backtest forgot to charge; it scales with turnover and the linear break-even hides it. The 20-seed control confirms the gross edge is genuinely there to *be* eaten — report gross *and* net with a turnover-aware cost model, or the backtest is meaningless.